# 自我反思记忆（Self-Reflection Memory）

> **每次任务尝试之后，智能体会停下来反思发生了什么，提取可长期复用的洞见（insight），并存入反思记忆。这样，同样的错误不会重犯，成功的策略也能跨会话延续。**

想象一位棋手，每场比赛结束后都会复盘。他们不只记录棋步，还会问：「我为什么会丢掉那颗子？我漏看了什么模式？」时间一长，这些赛后笔记比棋谱本身更有价值。

大多数记忆技术记录的是对话中发生了什么：消息、实体、摘要。自我反思记忆走另一条路。它记录的是事情为什么顺利或失败，以及下次该怎么做。这正是 **Reflexion**（Shinn et al., 2023）的核心思想：智能体通过语言层面的自我反思，把稀疏的结果信号变成丰富、可复用的学习。

这个模式分四步：
1. 智能体尝试一个任务，得到一个结果（成功、部分成功、失败）。
2. **反思提示词（reflection prompt）** 让智能体分析根本原因并提取经验教训。
3. 得到的**洞见**存入专门的反思记忆。
4. 在之后的任务开始前，智能体检索相关的过往反思，注入提示词。

这样就形成了**反思存储循环（reflect-then-store loop）**。这个循环让基于 LLM 的智能体（把语言模型当作推理引擎）在多次尝试之间持续进步，而且不需要任何参数更新。学习完全靠自然语言形式的反思来完成。

**读完本 notebook 你会理解：**
- 如何用结构化的反思提示词实现反思存储循环。
- 如何搭建一个支持持久化和检索的反思记忆库。
- 反思如何在多任务基准测试（一组标准化的测试任务）中提升智能体表现。
- 自我反思记忆与其他记忆技术相比的权衡。


## 核心概念

- **反思提示词（reflection prompt）：** 一组结构化的问题，引导智能体评估自己的表现。例如：*「哪些地方做得好？哪些失败了？根本原因是什么？下次该怎么做？」* 它把「通过/失败」这样的结果变成丰富的学习信号。
- **洞见提取（insight extraction）：** 把完整的反思提炼成一条简洁、可复用的原则。例如：*「处理日期转换前，先检查日期解析的边界情况。」* 洞见是反思记忆的存储单元。
- **反思记忆库（reflection memory store）：** 专门存放过往反思的记忆，按任务类型和结果建立索引。每个新任务开始前，智能体检索相关反思并注入上下文。
- **反思存储循环（reflect-then-store loop）：** 核心模式：尝试、评估结果、反思、提取洞见、存储、下次尝试时检索。每一轮循环都在积累智能体的经验。
- **结果评估（outcome evaluation）：** 把智能体的输出和预期结果对比，判定成功、部分成功或失败。这个信号触发反思。
- **Reflexion 框架：** 来自 Shinn et al.（2023）。智能体使用语言强化学习（verbal reinforcement learning，通过书面的自我反馈来学习）。自然语言反思取代梯度更新，成为学习机制。
- **元认知（meta-cognition）：** 智能体对自身推理过程进行推理的能力。自我反思记忆就是元认知监控（观察自己的思考过程）的一种显式实现。


## 架构

<p align="center">
  <img src="../../images/diagrams/16_self_reflection_memory.svg" alt="自我反思记忆架构图" width="720"/>
</p>

<details><summary>Mermaid 源码</summary>

```mermaid
flowchart LR
    subgraph Attempt["Task Attempt"]
        A["Task prompt"] --> B["Retrieve relevant\npast reflections"]
        B --> C["Agent executes\ntask (with reflections\nin context)"]
        C --> D["Output"]
    end

    subgraph Evaluate["Outcome Evaluation"]
        D --> E["Compare output\nvs. expected"]
        E --> F{"Success?"}
    end

    subgraph Reflect["Reflection Loop"]
        F -->|"fail / partial"| G["Reflection prompt:\nWhat went wrong?\nRoot cause?\nLesson learned?"]
        F -->|"success"| H["Reflection prompt:\nWhat worked well?\nKey strategy?"]
        G --> I["Generate structured\nreflection"]
        H --> I
        I --> J["Extract concise\ninsight"]
    end

    subgraph Store["Reflection Memory"]
        J --> K[("Reflection Store\n─────────────\ntask_type\noutcome\ninsight\ntimestamp")]
        K --> B
    end

    style K fill:#4f46e5,color:#fff
    style C fill:#059669,color:#fff
    style I fill:#d97706,color:#fff
    style F fill:#dc2626,color:#fff
```

</details>

## 环境准备

安装所需的包。这个单元格只需要运行一次。


In [ ]:
# 安装所需的包（只需运行一次）
%pip install -q python-dotenv matplotlib numpy deepagents langchain-openai

加载 Anthropic SDK 和标准库工具。
API key 来自 `.env` 文件。


In [ ]:
import os
import json
import time
from datetime import datetime

from dotenv import load_dotenv

load_dotenv()  # 从 .env 读取 LLM_API_KEY / LLM_BASE_URL / LLM_MODEL_NAME

from langchain_openai import ChatOpenAI
from deepagents import create_deep_agent

assert os.getenv("LLM_API_KEY"), "Set LLM_API_KEY in your .env file"

# 任何 OpenAI 兼容端点都可以：用官方 OpenAI 时省略 base_url 即可
llm = ChatOpenAI(
    model=os.getenv("LLM_MODEL_NAME", "gpt-4o-mini"),
    api_key=os.getenv("LLM_API_KEY"),
    base_url=os.getenv("LLM_BASE_URL"),
)



def to_text(content) -> str:
    """把字符串或内容块列表统一转成纯文本。"""
    if isinstance(content, str):
        return content
    return "".join(
        block.get("text", "") for block in content if isinstance(block, dict)
    )


print("✓ Model ready")


## 核心实现

我们的自我反思记忆系统由三个组件构成：

1. **`ReflectionStore`：** 存储结构化反思（任务类型、结果、洞见、完整反思文本），支持持久化和基于关键词的检索。
2. **`ReflectionGenerator`：** 用 Claude 分析任务结果，生成带洞见的结构化反思。
3. **`ReflectiveAgent`：** 主智能体，负责尝试任务、评估结果、在失败时触发反思，并检索过往反思来改进后续尝试。


In [ ]:
class ReflectionStore:
    """存储结构化反思，支持按任务类型和关键词匹配检索。"""

    def __init__(self):
        self.reflections: list[dict] = []

    def add(self, reflection: dict) -> None:
        """向存储中添加一条结构化反思。

        预期的键：task_type、outcome、insight、reflection_text、task_description
        """
        reflection["timestamp"] = datetime.now().isoformat()
        reflection["id"] = len(self.reflections)
        self.reflections.append(reflection)

    def retrieve(self, task_type: str | None = None, limit: int = 5) -> list[dict]:
        """检索反思，可按任务类型过滤。"""
        matches = self.reflections
        if task_type:
            matches = [r for r in matches if r.get("task_type") == task_type]
        # 最近的在前
        return sorted(matches, key=lambda r: r["timestamp"], reverse=True)[:limit]

    def retrieve_by_keywords(self, text: str, limit: int = 5) -> list[dict]:
        """检索洞见或 task_description 与文本有重合的反思。"""
        text_words = set(text.lower().split())
        scored = []
        for r in self.reflections:
            ref_text = f"{r.get('insight', '')} {r.get('task_description', '')}".lower()
            ref_words = set(ref_text.split())
            overlap = len(text_words & ref_words)
            if overlap > 0:
                scored.append((overlap, r))
        scored.sort(key=lambda x: x[0], reverse=True)
        return [r for _, r in scored[:limit]]

    def get_failure_insights(self, task_type: str | None = None) -> list[str]:
        """获取所有失败尝试的洞见，可按任务类型过滤。"""
        reflections = self.retrieve(task_type=task_type, limit=100)
        return [
            r["insight"] for r in reflections
            if r.get("outcome") == "failure" and r.get("insight")
        ]

    def format_for_prompt(self, reflections: list[dict]) -> str:
        """把反思格式化后注入智能体上下文。"""
        if not reflections:
            return "No relevant past reflections."
        lines = []
        for r in reflections:
            outcome = r.get("outcome", "unknown")
            icon = "\u2713" if outcome == "success" else "\u2717"
            lines.append(
                f"  {icon} [{outcome.upper()}] {r.get('task_description', 'N/A')}\n"
                f"    Insight: {r.get('insight', 'N/A')}"
            )
        return "\n".join(lines)

`ReflectionStore` 还需要持久化和工具方法。
下面添加 `save` 和 `load`，用于把反思写入 JSON 文件，
再加上便捷的 `__len__` 和 `__repr__`。


In [ ]:
def save(self, path: str) -> None:
    """把反思持久化到 JSON。"""
    with open(path, "w") as f:
        json.dump(self.reflections, f, indent=2)

ReflectionStore.save = save

def __len__(self) -> int:
    return len(self.reflections)

ReflectionStore.__len__ = __len__

def __repr__(self) -> str:
    n_fail = sum(1 for r in self.reflections if r.get("outcome") == "failure")
    n_ok = sum(1 for r in self.reflections if r.get("outcome") == "success")
    return f"ReflectionStore({len(self)} reflections: {n_ok} success, {n_fail} failure)"


print("\u2713 ReflectionStore class defined")
ReflectionStore.__repr__ = __repr__

print("\u2713 ReflectionStore class defined")

接下来定义用于生成反思的工具 schema。
工具 schema（tool schema）是一种结构化描述，告诉模型应该返回哪些字段。
这里让 Claude 返回四项输出：发生了什么、根本原因、简明的洞见，以及下次的策略。


In [ ]:
REFLECTION_TOOL = {
    "type": "function",
    "function": {
    "name": "store_reflection",
    "description": (
        "Store a structured reflection about a task attempt. "
        "Analyze what happened, identify root causes, and extract a concise insight."
    ),
        "parameters": {
        "type": "object",
        "properties": {
            "what_happened": {
                "type": "string",
                "description": "Brief summary of what the agent did and the outcome.",
            },
            "root_cause": {
                "type": "string",
                "description": "Why did the attempt succeed or fail? Identify the key factor.",
            },
            "insight": {
                "type": "string",
                "description": (
                    "A concise, reusable lesson learned (1-2 sentences). "
                    "Should be actionable and applicable to similar future tasks."
                ),
            },
            "strategy_for_next_time": {
                "type": "string",
                "description": "Concrete strategy to apply when facing a similar task.",
            },
        },
        "required": ["what_happened", "root_cause", "insight", "strategy_for_next_time"],
    },
    },
}


`ReflectionGenerator` 把任务结果连同上面的工具 schema 一起发给 Claude，
Claude 返回结构化的反思。
如果工具调用失败，生成器会返回一个带占位值的安全兜底结果。


In [ ]:
class ReflectionGenerator:
    """用工具调用从任务结果生成结构化反思。"""

    def __init__(self, model):
        # tool_choice="required" 强制模型必须调用工具
        self.model = model.bind_tools([REFLECTION_TOOL], tool_choice="required")


`reflect` 方法把任务结果发给 Claude，请求结构化分析。
它利用 Claude 的工具调用（tool use）能力，拿回一个干净的四字段 JSON 对象：
发生了什么、根本原因、洞见，以及下次的策略。


In [ ]:
def reflect(
    self,
    task_description: str,
    agent_output: str,
    expected_output: str,
    outcome: str,
) -> dict:
    """对一次任务尝试生成结构化反思。

    Args:
        task_description: 任务是什么。
        agent_output: 智能体产出了什么。
        expected_output: 预期是什么。
        outcome: 'success'、'failure' 或 'partial'。

    Returns:
        字典，键为：what_happened、root_cause、insight、strategy_for_next_time
    """
    response = self.model.invoke([
        {"role": "system", "content": (
            "You are a reflection system that analyzes task outcomes. "
            "Given a task, the agent's output, and the expected output, "
            "generate a thoughtful reflection. Focus on extractable lessons "
            "that would help in future similar tasks. Be specific and actionable."
        )},
        {"role": "user", "content": (
            f"Task: {task_description}\n\n"
            f"Agent output: {agent_output}\n\n"
            f"Expected output: {expected_output}\n\n"
            f"Outcome: {outcome}\n\n"
            "Analyze this attempt and extract a lesson learned."
        )},
    ])

    for call in response.tool_calls or []:
        if call["name"] == "store_reflection":
            return call["args"]
    return {
        "what_happened": "Reflection generation failed.",
        "root_cause": "Unknown",
        "insight": "No insight extracted.",
        "strategy_for_next_time": "Retry with more context.",
    }


ReflectionGenerator.reflect = reflect

print("✓ ReflectionGenerator class defined (uses tool calling)")


我们需要一种方法来判断智能体是否给出了正确答案。
这个评估器使用关键词匹配。
它在智能体的输出里查找预期关键词。
命中 80% 及以上为「成功」，30% 到 80% 为「部分成功」，低于 30% 为「失败」。


In [ ]:
# ── 任务评估器 ──

def evaluate_output(agent_output: str, expected: str, check_keywords: list[str]) -> str:
    """简单的关键词评估：检查输出里是否出现所需的关键词。

    全部命中返回 'success'，部分命中返回 'partial'，全未命中返回 'failure'。
    """
    output_lower = agent_output.lower()
    found = [kw for kw in check_keywords if kw.lower() in output_lower]
    ratio = len(found) / len(check_keywords) if check_keywords else 0
    if ratio >= 0.8:
        return "success"
    elif ratio >= 0.3:
        return "partial"
    return "failure"


这是我们的基准测试：六个任务，考察不同的技能。
每个任务包含一个提示词、一个预期答案和一组检查关键词。
这些任务故意设计了陷阱。
朴素的尝试往往栽在特定的坑上，而带着反思的尝试能从过往错误中学习。


In [ ]:
# ── 多任务基准测试 ──
# 每个任务包含：description、prompt、预期答案摘要和检查关键词。
# 任务经过设计：朴素的尝试往往在特定的坑上失败，
# 而带着反思的尝试能从过往错误中学习。

BENCHMARK_TASKS = [
    {
        "task_type": "date_parsing",
        "description": "Parse ambiguous date formats",
        "prompt": (
            "Parse the following date string and return it in ISO 8601 format (YYYY-MM-DD). "
            "The date is: '03/04/2025'. Assume the format is DD/MM/YYYY (European convention). "
            "Return ONLY the ISO date, nothing else."
        ),
        "expected": "2025-04-03",
        "check_keywords": ["2025-04-03"],
    },
    {
        "task_type": "list_processing",
        "description": "Extract and deduplicate items from text",
        "prompt": (
            "Extract all unique fruit names from this text and return them as a comma-separated "
            "list in alphabetical order:\n"
            "'I bought apples, then more apples, some bananas, three oranges, "
            "a kiwi, more bananas, and finally a mango.'\n"
            "Return ONLY the comma-separated list, nothing else."
        ),
        "expected": "apples, bananas, kiwi, mango, oranges",
        "check_keywords": ["apples", "bananas", "kiwi", "mango", "oranges"],
    },
    {
        "task_type": "code_generation",
        "description": "Write a function with specific edge case handling",
        "prompt": (
            "Write a Python function called `safe_divide(a, b)` that returns a/b. "
            "It must handle: division by zero (return 0), non-numeric inputs (return None), "
            "and both int and float inputs. Include a docstring. "
            "Return ONLY the function code, no explanation."
        ),
        "expected": "def safe_divide with zero check and type check",
        "check_keywords": ["def safe_divide", "zero", "none", "docstring"],
    },
]

print(f"Benchmark batch 1: {len(BENCHMARK_TASKS)} tasks")

剩下三个基准测试任务在这里。
它们考察文本转换、数据抽取和推理。
我们把两批任务合并成一个 `BENCHMARK_TASKS` 列表。


In [ ]:
BENCHMARK_TASKS_2 = [
    {
        "task_type": "reasoning",
        "description": "Multi-step logical reasoning",
        "prompt": (
            "A farmer has 3 fields. Field A produces 120 bushels/acre across 5 acres. "
            "Field B produces 95 bushels/acre across 8 acres. "
            "Field C produces 110 bushels/acre across 6 acres. "
            "Which field produces the MOST total bushels? "
            "Show your calculation for each field, then state the answer. "
            "Format: 'Field X: Y bushels' for each, then 'Answer: Field Z'."
        ),
        "expected": "Field B: 760 bushels. Answer: Field B",
        "check_keywords": ["760", "field b"],
    },
    {
        "task_type": "text_transformation",
        "description": "Apply a specific text transformation rule",
        "prompt": (
            "Convert the following sentence to title case, but keep articles "
            "(a, an, the), conjunctions (and, but, or), and prepositions "
            "(in, on, at, to, for, of, with) in lowercase UNLESS they start the sentence. "
            "Sentence: 'the quick brown fox jumps over the lazy dog in the park'\n"
            "Return ONLY the transformed sentence."
        ),
        "expected": "The Quick Brown Fox Jumps over the Lazy Dog in the Park",
        "check_keywords": ["Quick Brown Fox", "over the", "in the"],
    },
    {
        "task_type": "data_extraction",
        "description": "Extract structured data from unstructured text",
        "prompt": (
            "Extract all email addresses from this text and return them one per line, sorted alphabetically:\n"
            "'Contact us at support@acme.com or sales@acme.com. "
            "For technical issues, email dev@acme.com. "
            "Our CEO can be reached at alice@acme.com. "
            "Spam this address: noreply@acme.com'\n"
            "Return ONLY the email addresses, one per line, sorted alphabetically."
        ),
        "expected": "alice@acme.com\ndev@acme.com\nnoreply@acme.com\nsales@acme.com\nsupport@acme.com",
        "check_keywords": ["alice@acme.com", "dev@acme.com", "noreply@acme.com", "sales@acme.com", "support@acme.com"],
    },
]

BENCHMARK_TASKS.extend(BENCHMARK_TASKS_2)

print(f"\u2713 Benchmark defined: {len(BENCHMARK_TASKS)} tasks")
for t in BENCHMARK_TASKS:
    print(f"  \u2022 [{t['task_type']}] {t['description']}")

`ReflectiveAgent` 把所有组件串成反思存储循环。

每个任务开始前，它检索相关的过往反思并注入提示词。
每个任务结束后，它评估结果。
只要不是干脆利落的成功，它就生成一条新反思并存入。
多轮运行之后，智能体积累的洞见会帮它避免重蹈覆辙。


In [ ]:
class ReflectiveAgent:
    """尝试任务、对结果做反思、并借助存储的洞见持续改进的智能体。

    反思存储循环：
    1. 为当前任务检索相关的过往反思。
    2. 把反思作为上下文注入提示词。
    3. 尝试任务。
    4. 评估结果。
    5. 如果不是干脆利落的成功，生成反思并存入。
    6. 返回结果和元数据。
    """

    def __init__(
        self,
        model,
        reflection_store: ReflectionStore | None = None,
    ):
        self.model = model
        self.store = reflection_store or ReflectionStore()
        self.generator = ReflectionGenerator(model=model)
        self.attempt_log: list[dict] = []

`attempt_task` 方法是反思存储循环的心脏。
它检索过往反思、注入提示词、执行任务、评估结果，
并在结果不够好时生成一条新反思。


尝试任务之前，智能体会先收集过往反思。
这个辅助方法按任务类型和关键词重合度检索反思，
然后把它们格式化成上下文字符串，注入提示词。


In [ ]:
def _build_reflection_context(self, task, use_reflections):
    """检索相关反思并构建上下文字符串。"""
    reflections_used = []
    reflection_context = ""
    if use_reflections:
        by_type = self.store.retrieve(task_type=task["task_type"], limit=3)
        by_keywords = self.store.retrieve_by_keywords(task["prompt"], limit=2)
        seen_ids = set()
        merged = []
        for r in by_type + by_keywords:
            if r["id"] not in seen_ids:
                seen_ids.add(r["id"])
                merged.append(r)
        reflections_used = merged[:5]
        if reflections_used:
            formatted = self.store.format_for_prompt(reflections_used)
            reflection_context = (
                "\n\nYou have these insights from past attempts at similar tasks. "
                "Apply these lessons to avoid repeating mistakes:\n\n"
                f"{formatted}\n"
            )
    return reflections_used, reflection_context

ReflectiveAgent._build_reflection_context = _build_reflection_context

`attempt_task` 方法运行完整的反思存储循环。
它构建带反思上下文的提示词、执行任务、评估结果，
并在结果不是干脆利落的成功时存入一条新反思。


这个辅助方法负责任务执行、评估和反思生成。
它把提示词发给 Claude，用预期关键词检查输出，
并在结果不是干脆利落的成功时生成结构化反思。


In [ ]:
def _execute_and_reflect(self, task, reflections_used, reflection_context):
    """执行任务，评估结果，并在需要时生成反思。"""
    system = (
        "You are a precise task executor. Follow instructions exactly. "
        "Pay close attention to output format requirements."
        f"{reflection_context}"
    )

    response = self.model.invoke([
        {"role": "system", "content": system},
        {"role": "user", "content": task["prompt"]},
    ])
    output = to_text(response.content)

    outcome = evaluate_output(output, task["expected"], task["check_keywords"])

    reflection = None
    if outcome != "success":
        reflection = self.generator.reflect(
            task_description=task["description"],
            agent_output=output,
            expected_output=task["expected"],
            outcome=outcome,
        )
        self.store.add({
            "task_type": task["task_type"],
            "task_description": task["description"],
            "outcome": outcome,
            "insight": reflection.get("insight", ""),
            "reflection_text": json.dumps(reflection),
        })

    return output, outcome, reflection

ReflectiveAgent._execute_and_reflect = _execute_and_reflect

`attempt_task` 方法把这些部件串起来。
它检索过往反思，带着上下文执行任务，并记录结果。


In [ ]:
def attempt_task(self, task, use_reflections=True):
    """尝试一个任务，可选地用过往反思做指导。

    Args:
        task: 字典，键为 task_type、description、prompt、expected、check_keywords
        use_reflections: 是否检索并注入过往反思。

    Returns:
        字典，包含：output、outcome、reflection（如果生成了）、reflections_used
    """
    reflections_used, reflection_context = self._build_reflection_context(
        task, use_reflections,
    )

    output, outcome, reflection = self._execute_and_reflect(
        task, reflections_used, reflection_context,
    )

    result = {
        "task_type": task["task_type"],
        "description": task["description"],
        "output": output,
        "expected": task["expected"],
        "outcome": outcome,
        "reflection": reflection,
        "reflections_used": len(reflections_used),
    }
    self.attempt_log.append(result)
    return result

ReflectiveAgent.attempt_task = attempt_task

剩下的方法负责基准测试和持久化。
`run_benchmark` 运行完整的任务套件并打印摘要。
`save_memory` 和 `from_saved` 负责跨会话写入和加载反思。


In [ ]:

def run_benchmark(
    self,
    tasks: list[dict],
    use_reflections: bool = True,
    label: str = "Run",
) -> dict:
    """运行完整的基准测试套件，返回汇总结果。"""
    results = []
    for i, task in enumerate(tasks):
        result = self.attempt_task(task, use_reflections=use_reflections)
        status = "\u2713" if result["outcome"] == "success" else (
            "~" if result["outcome"] == "partial" else "\u2717"
        )
        ref_info = f" (used {result['reflections_used']} reflections)" if use_reflections else ""
        print(f"  {status} [{result['task_type']}] {result['description']}{ref_info}")
        results.append(result)

    n_success = sum(1 for r in results if r["outcome"] == "success")
    n_partial = sum(1 for r in results if r["outcome"] == "partial")
    n_failure = sum(1 for r in results if r["outcome"] == "failure")

    summary = {
        "label": label,
        "total": len(results),
        "success": n_success,
        "partial": n_partial,
        "failure": n_failure,
        "score": n_success / len(results) if results else 0,
        "results": results,
    }
    print(f"\n  Score: {n_success}/{len(results)} "
          f"({n_success/len(results)*100:.0f}%) success | "
          f"{n_partial} partial | {n_failure} failure")
    return summary
ReflectiveAgent.run_benchmark = run_benchmark

def save_memory(self, path: str) -> None:
    """把反思存储持久化到磁盘。"""
    self.store.save(path)
ReflectiveAgent.save_memory = save_memory

@classmethod
def from_saved(cls, path: str, **kwargs) -> "ReflectiveAgent":
    """创建一个预加载了已保存反思的智能体。"""
    store = ReflectionStore.load(path)
    return cls(reflection_store=store, **kwargs)
ReflectiveAgent.from_saved = from_saved

def __repr__(self) -> str:
    return f"ReflectiveAgent(reflections={len(self.store)}, attempts={len(self.attempt_log)})"
ReflectiveAgent.__repr__ = __repr__

print("\u2713 ReflectiveAgent class defined")

## 用法示例：反思存储循环

我们用一个任务走一遍反思存储循环。你会看清反思的每一步。智能体会：
1. 尝试任务。
2. 评估结果。
3. 如果失败，生成带根因分析和可行动洞见的反思。
4. 把洞见存起来供将来使用。


In [ ]:
# 创建一个全新的智能体
agent = ReflectiveAgent(model=llm)

# 一个容易在细节上出错的任务
demo_task = {
    "task_type": "date_parsing",
    "description": "Parse ambiguous date formats",
    "prompt": (
        "Parse the following date string and return it in ISO 8601 format (YYYY-MM-DD). "
        "The date is: '03/04/2025'. Assume the format is DD/MM/YYYY (European convention). "
        "Return ONLY the ISO date, nothing else."
    ),
    "expected": "2025-04-03",
    "check_keywords": ["2025-04-03"],
}

# 第一次尝试：还没有任何反思可用
print("=== Attempt 1 (no prior reflections) ===\n")
result1 = agent.attempt_task(demo_task, use_reflections=True)
print(f"  Output:   {result1['output'].strip()}")
print(f"  Expected: {result1['expected']}")
print(f"  Outcome:  {result1['outcome']}")

if result1["reflection"]:
    print(f"\n  \U0001f4ad Reflection generated:")
    print(f"     What happened:  {result1['reflection']['what_happened']}")
    print(f"     Root cause:     {result1['reflection']['root_cause']}")
    print(f"     Insight:        {result1['reflection']['insight']}")
    print(f"     Next time:      {result1['reflection']['strategy_for_next_time']}")

print(f"\n  Reflections in store: {len(agent.store)}")

# 第二次尝试：此时已有第一次的反思可用
print("\n=== Attempt 2 (with reflection from attempt 1) ===\n")
result2 = agent.attempt_task(demo_task, use_reflections=True)
print(f"  Output:   {result2['output'].strip()}")
print(f"  Expected: {result2['expected']}")
print(f"  Outcome:  {result2['outcome']}")
print(f"  Reflections used: {result2['reflections_used']}")

print(f"\n\u2713 Reflect-then-store loop demonstrated")

## 多任务基准测试：反思有用吗？

现在做一个对照实验。我们运行同一个 6 任务基准测试 **三次**：

1. **第 1 轮（基线）：** 没有任何反思。智能体冷启动完成所有任务。
2. **第 2 轮（带反思）：** 智能体已有第 1 轮失败产生的反思。过往洞见被注入提示词。
3. **第 3 轮（累积）：** 智能体已有前两轮的反思，进一步改进自己的做法。

如果自我反思记忆有效，随着智能体从错误中学习，分数应该在几轮之间**逐步上升**。


In [ ]:
# 为实验创建全新的智能体
agent = ReflectiveAgent(model=llm)

# ── 第 1 轮：基线（还没有反思）──
print("=" * 60)
print("RUN 1 - Baseline (no reflections available)")
print("=" * 60)
run1 = agent.run_benchmark(BENCHMARK_TASKS, use_reflections=True, label="Run 1 (Baseline)")

print(f"\nReflections after Run 1: {len(agent.store)}")
print()

# ── 第 2 轮：带着第 1 轮的反思 ──
print("=" * 60)
print("RUN 2 - With reflections from Run 1")
print("=" * 60)
run2 = agent.run_benchmark(BENCHMARK_TASKS, use_reflections=True, label="Run 2 (Reflective)")

print(f"\nReflections after Run 2: {len(agent.store)}")
print()

# ── 第 3 轮：带着累积的反思 ──
print("=" * 60)
print("RUN 3 - With accumulated reflections from Runs 1 & 2")
print("=" * 60)
run3 = agent.run_benchmark(BENCHMARK_TASKS, use_reflections=True, label="Run 3 (Accumulated)")

print(f"\nReflections after Run 3: {len(agent.store)}")

为了公平比较，我们再跑一次同样的基准测试，但禁用反思。
这个对照组用来验证：存储的反思是否真的帮智能体表现得更好。


In [ ]:
# ── 对照组：同样的基准测试，但不使用反思 ──
print("=" * 60)
print("CONTROL - Same agent, reflections DISABLED")
print("=" * 60)

# 创建一个新智能体，存储的反思相同，但禁用检索
control_agent = ReflectiveAgent(model=llm)
control = control_agent.run_benchmark(
    BENCHMARK_TASKS, use_reflections=False, label="Control (No Reflection)"
)

我们把结果可视化。
左图是四轮运行的总体得分。
右图是逐任务的热力图：绿色代表成功，黄色代表部分成功，红色代表失败。
看看哪些任务在几轮之间从红色变成了绿色。


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── 面板 1：各轮得分变化 ──
runs = [run1, run2, run3, control]
labels = [r["label"] for r in runs]
scores = [r["success"] for r in runs]
colors = ["#94a3b8", "#4f46e5", "#059669", "#ef4444"]

bars = axes[0].bar(labels, scores, color=colors, width=0.55, alpha=0.9)
axes[0].set_ylabel("Tasks Passed", fontsize=12)
axes[0].set_ylim(0, len(BENCHMARK_TASKS) + 1)
axes[0].set_title("Score Progression Across Benchmark Runs", fontsize=13, fontweight="bold")
axes[0].axhline(y=len(BENCHMARK_TASKS), color="gray", linestyle="--", alpha=0.3)
for bar, score in zip(bars, scores):
    axes[0].text(
        bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.2,
        f"{score}/{len(BENCHMARK_TASKS)}", ha="center", fontweight="bold", fontsize=12,
    )
axes[0].tick_params(axis="x", labelsize=9)

# ── 面板 2：逐任务提升热力图 ──
task_labels = [t["task_type"] for t in BENCHMARK_TASKS]
run_labels = ["Run 1", "Run 2", "Run 3"]
heatmap_data = []
for run in [run1, run2, run3]:
    row = []
    for result in run["results"]:
        val = 1.0 if result["outcome"] == "success" else (0.5 if result["outcome"] == "partial" else 0.0)
        row.append(val)
    heatmap_data.append(row)

heatmap_data = np.array(heatmap_data)
im = axes[1].imshow(heatmap_data, cmap="RdYlGn", aspect="auto", vmin=0, vmax=1)

axes[1].set_xticks(range(len(task_labels)))
axes[1].set_xticklabels(task_labels, rotation=45, ha="right", fontsize=9)
axes[1].set_yticks(range(len(run_labels)))
axes[1].set_yticklabels(run_labels, fontsize=10)
axes[1].set_title("Per-Task Outcomes Across Runs", fontsize=13, fontweight="bold")

# 添加文本标注
for i in range(len(run_labels)):
    for j in range(len(task_labels)):
        val = heatmap_data[i, j]
        text = "\u2713" if val == 1.0 else ("~" if val == 0.5 else "\u2717")
        axes[1].text(j, i, text, ha="center", va="center", fontsize=14,
                     color="white" if val < 0.5 else "black", fontweight="bold")

plt.tight_layout()
plt.savefig("reflection_benchmark.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\nRun 1 (Baseline):     {run1['success']}/{run1['total']}")
print(f"Run 2 (Reflective):   {run2['success']}/{run2['total']}")
print(f"Run 3 (Accumulated):  {run3['success']}/{run3['total']}")
print(f"Control (No Reflect): {control['success']}/{control['total']}")

## 检查反思存储

我们看看智能体学到了什么。下面是它在失败尝试之后实际生成并存储的反思。


In [ ]:
print(f"=== Reflection Store: {len(agent.store)} reflections ===\n")

for r in agent.store.reflections:
    outcome = r.get("outcome", "unknown")
    icon = "\u2713" if outcome == "success" else "\u2717"
    print(f"{icon} [{outcome.upper()}] {r['task_description']}")
    print(f"  Task type: {r['task_type']}")
    print(f"  Insight:   {r['insight']}")
    reflection_data = json.loads(r["reflection_text"])
    print(f"  Strategy:  {reflection_data.get('strategy_for_next_time', 'N/A')}")
    print()

## 跨会话持久化

反思会持久化到 JSON（一种标准的文本数据格式）。新的智能体实例可以加载过往会话积累的经验，立即从中受益。不需要在同样的错误上重新学习一遍。


In [ ]:
# 保存当前会话的反思
REFLECTIONS_FILE = "reflections.json"
agent.save_memory(REFLECTIONS_FILE)
print(f"\u2713 Saved {len(agent.store)} reflections to {REFLECTIONS_FILE}")

# 模拟一个新会话
print("\n--- NEW SESSION ---\n")
agent2 = ReflectiveAgent.from_saved(REFLECTIONS_FILE)
print(f"Loaded: {agent2.store}")

# 验证过往洞见可用
print(f"\nFailure insights for 'date_parsing':")
for insight in agent2.store.get_failure_insights("date_parsing"):
    print(f"  \u2022 {insight}")

print(f"\nFailure insights for 'code_generation':")
for insight in agent2.store.get_failure_insights("code_generation"):
    print(f"  \u2022 {insight}")

# 用加载的反思运行一个任务
print("\n=== New Session Task Attempt ===\n")
result = agent2.attempt_task(BENCHMARK_TASKS[0], use_reflections=True)
print(f"  Task:     {result['description']}")
print(f"  Output:   {result['output'].strip()}")
print(f"  Outcome:  {result['outcome']}")
print(f"  Reflections used: {result['reflections_used']}")
print(f"\n\u2713 Cross-session persistence works -- insights preserved!")

## 讨论与权衡

### 优点
- **无需微调（fine-tuning）就能学习：** 智能体只靠自然语言反思就能在多次尝试之间进步。没有梯度更新，不改模型。任何 LLM API 都适用。
- **显式且可检查：** 每条反思都是人类可读的。你能准确审计智能体学到了什么。这一点不同于不透明的参数更新或基于嵌入的记忆。
- **可组合：** 自我反思记忆可以叠加在任何其他记忆技术之上。结合实体记忆、知识图谱或向量库，可以构成多层架构。
- **高性价比的改进：** 每次失败一次反思 API 调用，就能避免同样的失败在未来所有会话中重演。反思调用的投资回报非常高。

### 缺点
- **反思质量取决于模型。** 如果 LLM 生成的反思肤浅或错误，存储的洞见可能产生误导。垃圾进，垃圾出。
- **每次失败都有额外 API 成本：** 每个失败的任务都会触发一次额外的 LLM 调用来生成反思。在高失败率场景下，这笔开销会累积。
- **关键词检索很脆弱：** 我们的关键词匹配检索可能漏掉相关反思，也可能捞出无关的。生产系统应该改用基于嵌入的相似度（用向量运算比较语义）。
- **没有矛盾处理：** 如果智能体在几轮之间生成了相互冲突的洞见，两条都会被存储。生产系统需要冲突检测和解决机制。
- **评估是瓶颈：** 整个系统依赖能否评估结果。没有明确成功标准的任务无法触发有意义的反思。

### 自我反思记忆与其他方法对比

| 维度 | 自我反思记忆 | 摘要记忆 | 实体记忆 |
|--------|----------------------|----------------|---------------|
| 存什么 | 事情发生的*原因* | 发生了*什么* | 关于实体的事实 |
| 学习方式 | 在多次尝试间改进 | 压缩历史 | 跟踪状态 |
| 触发条件 | 任务结果（成功/失败） | 每 N 轮 | 提及实体 |
| 检索方式 | 按任务类型 + 关键词 | 始终注入 | 按实体名 |
| 元认知 | 是（对推理本身做推理） | 否 | 否 |
| 成本 | 仅失败时额外调用 | 每 N 轮额外调用 | 每轮额外调用 |

### 什么时候使用自我反思记忆

| 场景 | 建议 |
|----------|---------------|
| 执行可重复任务且有明确成功标准的智能体 | **极佳**（核心用例） |
| 需要有可证明进步的长期运行智能体 | **极佳** |
| 跨会话调试智能体失败 | **很好**（反思能解释失败*为什么*发生） |
| 问答聊天机器人 | **大材小用**（摘要或滑动窗口就够了） |
| 没有可评估结果的任务 | **不适用**（反思需要结果信号） |
| 成本敏感且失败率低 | **不错**（反思只在失败时触发） |


## 延伸阅读

- [Shinn et al., "Reflexion: Language Agents with Verbal Reinforcement Learning" (2023)](https://arxiv.org/abs/2303.11366)。LLM 智能体自我反思的奠基论文。
- [Madaan et al., "Self-Refine: Iterative Refinement with Self-Feedback" (2023)](https://arxiv.org/abs/2303.17651)。相关方法，用自我反馈做迭代改进。
- [Yao et al., "ReAct: Synergizing Reasoning and Acting" (2023)](https://arxiv.org/abs/2210.03629)。与反思互补的推理轨迹。
- [Anthropic Tool Use (Function Calling)](https://docs.anthropic.com/en/docs/build-with-claude/tool-use?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques)。用于结构化反思提取的 API。
- [Flavell, "Metacognition and Cognitive Monitoring" (1979)](https://doi.org/10.1037/0003-066X.34.10.906)。关于元认知的认知科学奠基工作。
- [Anthropic: Building Conversational AI](https://docs.anthropic.com/en/docs/build-with-claude/conversational-ai?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques)。多轮对话模式。

---

*← 上一篇：[15：记忆压缩](../15_memory_compaction/) · 下一篇：[17：记忆路由](../17_memory_routing/) →*


删除本演示过程中产生的临时文件。


In [ ]:
# 清理临时文件
import os
for f in ["reflections.json", "reflection_benchmark.png"]:
    if os.path.exists(f):
        os.remove(f)
        print(f"Removed {f}")

## 🧪 动手练习

三个小挑战，帮你加深理解。每个预计花 10-30 分钟。

### 挑战 1：反思分类
扩展 `ReflectionStore`，给每条反思打上类别标签：'strategy'、'mistake' 或 'preference'。修改 `retrieve()` 接受一个可选的类别过滤参数。测试方法：运行 `run_benchmark()`，在下一次尝试时只查询与错误相关的反思。

### 挑战 2：测量学习曲线
在同一个任务集上连续运行 `run_benchmark()` 5 轮。每轮结束后记录成功率。画出学习曲线（轮次对成功率）。算一算智能体经过多少轮后趋于平稳。

### 挑战 3：智能体之间共享反思
创建两个处理不同任务类型的 `ReflectiveAgent` 实例。两者都完成基准测试后，从一个智能体导出反思，导入另一个。测量跨智能体的反思是否能提升表现。这与 22 多智能体共享记忆中的共享记忆模式相关。


![](https://europe-west1-amt-views-tracker.cloudfunctions.net/amt-tracker?notebook=all-techniques--16-self-reflection-memory--self-reflection-memory)
